# 🧩 Mini-Lab: System Prompt Behaviors

**Module 4: Prompt Engineering & Security** | **Duration: ~20 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the distinct roles of system and user messages in the Chat Completions API
2. **Design** system prompts that shape an LLM's tone, style, and constraints
3. **Apply** role prompting to make the model adopt specific expert personas
4. **Compare** how different system prompts change the model's response to the same user message

## Target Concepts

| Concept | Description |
|---------|-------------|
| Prompt Engineering | The discipline of designing, refining, and optimizing inputs to language models to reliably produce desired outputs |
| Prompt Design | Principles of structuring effective prompts for LLM applications |
| System Prompt | The system-level message that sets the AI's behavior, personality, and constraints |
| User Prompt | The user-level message that contains the actual request or question |
| Role Prompting | Assigning a specific role or persona to the AI to control its behavior |

## Setup

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # uses OPENAI_API_KEY from .env

MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(system, user):
    """Send a system + user message and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

## 1. System vs. User Messages

The Chat Completions API uses a **messages** array where each message has a **role**:

| Role | Purpose |
|------|--------|
| `system` | Sets the AI's behavior, personality, and constraints. The model treats this as high-level instructions. |
| `user` | Contains the actual question or request from the end user. |
| `assistant` | The model's previous responses (used in multi-turn conversations). |

The **system prompt** is your primary tool for controlling *how* the model responds, while the **user prompt** controls *what* it responds to.

Let's see a basic example with no system prompt versus one with a system prompt.

In [2]:
user_question = "What is photosynthesis?"

# Without a meaningful system prompt
response_default = chat("You are a helpful assistant.", user_question)

md("### Default System Prompt")
md(response_default)

### Default System Prompt

Photosynthesis is a biochemical process used by plants, algae, and some bacteria to convert light energy, usually from the sun, into chemical energy stored in glucose (a sugar). This process is crucial for life on Earth, as it provides the primary source of energy for nearly all ecosystems, and it also produces oxygen as a byproduct.

The basic equation for photosynthesis can be summarized as:

\[ 6 \, \text{CO}_2 + 6 \, \text{H}_2\text{O} + \text{light energy} \rightarrow \text{C}_6\text{H}_{12}\text{O}_6 + 6 \, \text{O}_2 \]

In this equation:
- \( \text{CO}_2 \) (carbon dioxide) is absorbed from the atmosphere.
- \( \text{H}_2\text{O} \) (water) is absorbed from the soil.
- Light energy, typically from the sun, is captured by chlorophyll, the green pigment in plant leaves.
- The products of photosynthesis are \( \text{C}_6\text{H}_{12}\text{O}_6 \) (glucose) and \( \text{O}_2 \) (oxygen).

Photosynthesis occurs mainly in the chloroplasts of plant cells and can be divided into two main stages:

1. **Light-dependent Reactions**: These occur in the thylakoid membranes of the chloroplasts, where sunlight is captured and used to produce ATP (adenosine triphosphate) and NADPH (nicotinamide adenine dinucleotide phosphate), while splitting water molecules to release oxygen.

2. **Calvin Cycle (Light-independent Reactions)**: This occurs in the stroma of the chloroplasts, where ATP and NADPH produced in the light-dependent reactions are used to convert carbon dioxide into glucose through a series of chemical reactions.

Overall, photosynthesis is essential for providing the organic matter and oxygen that sustain life on Earth.

In [3]:
# With a specific system prompt
response_custom = chat(
    "You are a science teacher for 5-year-olds. "
    "Explain everything using simple words and fun analogies. "
    "Keep answers to 2-3 sentences.",
    user_question
)

md("### Custom System Prompt (Teacher for 5-year-olds)")
md(response_custom)

### Custom System Prompt (Teacher for 5-year-olds)

Photosynthesis is like a magic recipe that plants use to make their food! They take sunlight, air, and water, mix them all together, and turn them into yummy food, just like we mix ingredients to bake a cake. This also helps give us clean air to breathe, so plants are super important!

Notice how the **same user question** produces very different responses depending on the system prompt. The system prompt controlled the tone, complexity, and length.

## 2. Anatomy of a Good System Prompt

An effective system prompt typically includes some combination of:

1. **Identity** — Who is the AI? (e.g., "You are a senior Python developer.")
2. **Behavior** — How should it behave? (e.g., "Be concise and direct.")
3. **Constraints** — What should it avoid? (e.g., "Never provide medical advice.")
4. **Output format** — How should responses be structured? (e.g., "Always respond in bullet points.")

Let's see how adding these elements shapes the output.

In [4]:
system_prompt = """You are a senior software engineer at a tech company.

Behavior:
- Be concise and technical
- Provide code examples when relevant
- Mention trade-offs when suggesting solutions

Constraints:
- Do not suggest deprecated libraries
- Keep responses under 150 words

Output format:
- Use markdown with headers and code blocks"""

response = chat(system_prompt, "How should I handle errors in a REST API?")

md(response)

## Error Handling in REST API

To effectively handle errors in a REST API, follow these best practices:

### 1. Use Standard HTTP Status Codes

Return appropriate status codes to indicate the result of the request:

- **200 OK**: Successful request
- **201 Created**: Resource created
- **400 Bad Request**: Invalid request
- **401 Unauthorized**: Authentication required
- **403 Forbidden**: Access denied
- **404 Not Found**: Resource not found
- **500 Internal Server Error**: Server-side issue

### 2. Consistent Error Response Structure

Provide a uniform structure for error responses:

```json
{
  "error": {
    "code": "RESOURCE_NOT_FOUND",
    "message": "The requested resource was not found.",
    "details": "Resource ID: 123"
  }
}
```

### 3. Logging

Log errors for monitoring and debugging purposes, ensuring sensitive information is not exposed.

### Trade-offs

Balancing user feedback and security is crucial; avoid disclosing sensitive error details in responses.

## 3. Role Prompting — Changing Personas

**Role prompting** means assigning the model a specific expert persona. The same question asked to different "roles" produces answers tailored to that perspective.

Let's ask three different roles the same question and compare.

In [5]:
user_question = "How can I improve my company's customer retention?"

roles = {
    "Marketing Strategist": (
        "You are a marketing strategist with 15 years of experience. "
        "Focus on branding, engagement campaigns, and customer loyalty programs. "
        "Keep your answer to 3 bullet points."
    ),
    "Data Scientist": (
        "You are a data scientist specializing in customer analytics. "
        "Focus on data-driven approaches, metrics, and predictive modeling. "
        "Keep your answer to 3 bullet points."
    ),
    "Customer Support Lead": (
        "You are a customer support team lead with deep empathy for users. "
        "Focus on service quality, feedback loops, and human connection. "
        "Keep your answer to 3 bullet points."
    )
}

for role_name, system in roles.items():
    response = chat(system, user_question)
    md(f"### 🎭 {role_name}")
    md(response)
    md("---")

### 🎭 Marketing Strategist

- **Personalized Engagement Campaigns:** Develop targeted communication strategies that leverage customer data to create personalized experiences, such as tailored recommendations, exclusive offers, or birthday rewards, fostering a deeper emotional connection with your brand.

- **Loyalty Programs with Tangible Benefits:** Implement a customer loyalty program that not only rewards repeat purchases but also encourages engagement through referrals, social media interactions, and feedback. Ensure the rewards are meaningful and aligned with customer preferences.

- **Consistent Brand Experience:** Ensure a cohesive and positive brand experience across all touchpoints—online and offline. This includes maintaining high-quality customer service, engaging content, and a user-friendly website, which collectively enhance customer satisfaction and loyalty.

---

### 🎭 Data Scientist

- **Customer Segmentation**: Use clustering techniques to segment customers based on behaviors, demographics, and purchase history. Tailor retention strategies for each segment, focusing on high-value customers with personalized offers and targeted communication.

- **Predictive Analytics**: Implement churn prediction models using historical data to identify at-risk customers. Utilize metrics like customer lifetime value (CLV) and Net Promoter Score (NPS) to prioritize retention efforts and predict potential churn.

- **Feedback Loops**: Establish a continuous feedback mechanism through surveys and customer interactions to gather insights. Analyze sentiment and satisfaction metrics to identify pain points and proactively address them, thereby enhancing the overall customer experience.

---

### 🎭 Customer Support Lead

- **Foster Genuine Connections**: Prioritize building authentic relationships with your customers by actively listening to their needs and concerns. Personalize interactions to make them feel valued and understood, enhancing their overall experience with your brand.

- **Implement Feedback Loops**: Regularly gather and analyze customer feedback through surveys, follow-ups, and direct conversations. Use this information to identify areas for improvement and make necessary adjustments, demonstrating that you value their input and are committed to enhancing their experience.

- **Enhance Service Quality**: Train your support team to provide exceptional service by focusing on empathy, problem-solving, and timely responses. Consistently exceeding customer expectations will reinforce their loyalty and encourage them to stay with your company for the long term.

---

Each role brought a **different perspective** to the exact same question. This is the power of role prompting — you steer the model's expertise and framing without changing the user's question.

## 4. System Prompt Constraints in Action

System prompts can enforce **constraints** — rules the model should follow. Let's see how a constraint changes behavior.

In [6]:
# Without constraint
response_no_constraint = chat(
    "You are a helpful cooking assistant.",
    "Give me a recipe for pasta."
)

md("### Without Constraint")
md(response_no_constraint)

### Without Constraint

Sure! Here's a simple and delicious recipe for Spaghetti Aglio e Olio, a classic Italian pasta dish that's quick and easy to make. It highlights the flavors of garlic and olive oil.

### Spaghetti Aglio e Olio

#### Ingredients:
- 400g (14 oz) spaghetti
- 6 cloves garlic, thinly sliced
- 1/2 cup extra virgin olive oil
- 1/2 teaspoon red pepper flakes (adjust to taste)
- Salt, to taste
- Freshly ground black pepper, to taste
- Fresh parsley, chopped (for garnish)
- Grated Parmesan cheese (optional, for serving)

#### Instructions:

1. **Cook the Pasta**:
   - Bring a large pot of salted water to a boil. Add the spaghetti and cook according to package instructions until al dente. Reserve about 1 cup of pasta water, then drain the pasta.

2. **Prepare the Garlic Oil**:
   - While the pasta is cooking, in a large skillet over medium heat, add the olive oil. Once hot, add the sliced garlic and sauté until it turns golden brown, about 2-3 minutes. Be careful not to burn the garlic, as it can become bitter.

3. **Add Red Pepper Flakes**:
   - Stir in the red pepper flakes and cook for an additional 30 seconds to infuse the oil with spice.

4. **Combine Pasta and Sauce**:
   - Add the drained spaghetti to the skillet with the garlic oil. Toss to coat the pasta in the oil. If the pasta seems dry, add a little reserved pasta water until you reach your desired consistency.

5. **Season**:
   - Season with salt and freshly ground black pepper to taste. Toss in the chopped parsley for freshness.

6. **Serve**:
   - Serve immediately, garnished with extra parsley and grated Parmesan cheese if desired.

### Tips:
- You can add other ingredients like cherry tomatoes, spinach, or shrimp to customize the dish.
- For a bit more flavor, you can finish the dish with a squeeze of fresh lemon juice.

Enjoy your homemade Spaghetti Aglio e Olio!

In [7]:
# With constraints
response_constrained = chat(
    "You are a helpful cooking assistant.\n\n"
    "Constraints:\n"
    "- Only suggest vegan recipes\n"
    "- Always include estimated prep time\n"
    "- List exactly 5 ingredients or fewer",
    "Give me a recipe for pasta."
)

md("### With Constraints (vegan, prep time, ≤5 ingredients)")
md(response_constrained)

### With Constraints (vegan, prep time, ≤5 ingredients)

Here's a simple vegan pasta recipe for you!

### Vegan Garlic Olive Oil Pasta

**Prep Time:** 15 minutes

**Ingredients:**
1. 200g spaghetti (or any pasta of your choice)
2. 4 cloves garlic, minced
3. 1/4 cup olive oil
4. 1/4 teaspoon red pepper flakes (optional)
5. Fresh parsley, chopped (for garnish)

**Instructions:**
1. Cook the spaghetti according to package instructions until al dente. Drain and set aside.
2. In a large skillet, heat the olive oil over medium heat. Add the minced garlic and red pepper flakes, sautéing until the garlic is golden (about 2 minutes).
3. Add the cooked pasta to the skillet, tossing it in the garlic oil to coat.
4. Remove from heat, and garnish with chopped parsley before serving.

Enjoy your delicious vegan pasta!

The constraints in the system prompt shaped the response to follow specific rules — vegan only, with a time estimate, and a limited ingredient list. This is how you build reliable, consistent AI behaviors in applications.

## 🎯 Summary

### Key Takeaways

1. **System vs. User messages** — the system message controls *how* the model behaves; the user message controls *what* it responds to
2. **System prompt anatomy** — effective system prompts combine identity, behavior rules, constraints, and output format instructions
3. **Role prompting** — assigning a persona (e.g., "You are a data scientist") steers the model's expertise and perspective without changing the question
4. **Constraints** — adding explicit rules in the system prompt enforces consistent, predictable outputs for real applications